# Extract Entities

List all entities (people, places, objects, brands, concepts) found across your video collection.
Use this for building a content index, understanding who and what appears in your videos, or feeding entity lists to downstream classification systems.

In [ ]:
import json
import os

import requests

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
BASE_URL = "https://api.twelvelabs.io/v1.3"
HEADERS = {"x-api-key": API_KEY, "Content-Type": "application/json"}

# Replace with your knowledge store ID
STORE_ID = "your_knowledge_store_id"

## Helper Functions

A utility to extract text content from a Jockey API response.

In [ ]:
def parse_response(result: dict) -> str | dict:
    """Extract text content from a Jockey API response."""
    for output in result["output"]:
        if output["type"] == "message":
            for content in output["content"]:
                return content["text"]
    return ""

## Entity Schema

Define a JSON schema for structured entity extraction. Each entity includes its name, type
(person, place, object, brand, or concept), frequency of appearance, and which videos it appears in.

In [ ]:
ENTITY_SCHEMA = {
    "type": "object",
    "properties": {
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "type": {"type": "string"},
                    "frequency": {"type": "string"},
                    "appears_in": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                },
            },
        },
        "entity_count": {"type": "integer"},
    },
}

## Extract All Entities

Ask Jockey to scan every video in the knowledge store and return a comprehensive entity list.
The prompt requests all distinct entities with their frequency information.

In [ ]:
response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": (
                    "List every distinct entity across all videos -- people, places, "
                    "objects, brands, and concepts. Include how frequently each appears."
                ),
            }
        ],
        "knowledge_store_id": STORE_ID,
        "text": {"format": {"type": "json_schema", "name": "entity_list", "schema": ENTITY_SCHEMA}},
    },
)

result = response.json()
data = json.loads(parse_response(result))

print(f"Found {data['entity_count']} entities:\n")
for entity in data["entities"]:
    print(f"  [{entity['type']}] {entity['name']} -- {entity['frequency']}")
    if entity.get("appears_in"):
        print(f"    Videos: {', '.join(entity['appears_in'])}")

## Variations

Modify the prompt to tailor entity extraction to your needs:

- **Filter by type:** "List only the people who appear in these videos"
- **Cross-video tracking:** "Which entities appear in more than one video?"
- **With relationships:** "List entities and how they relate to each other"

## Next Steps

- **[Get Corpus Overview](get_corpus_overview.ipynb)** -- understand the full collection context
- **[Search Videos](search_videos.ipynb)** -- find specific moments by description
- **[Find Organization Axes](find_organization_axes.ipynb)** -- discover the best categorization strategies
- **[Enrich Content](enrich_content.ipynb)** -- get deeper, domain-specific metadata

See also:
- [Structured Output Guide](../../docs/guides/structured-output.md) -- more on JSON schema responses